# Frankie NSFW Generator — ComfyUI + RealVis XL 5.0 + IPAdapter FaceID

**Photoreal** NSFW-capable SDXL checkpoint with face lock from Frankie reference.

Runtime → **Run all** → ~10 min → 8 uncensored photoreal Frankie selfies.

In [ ]:
!nvidia-smi | head -15

In [ ]:
import os
if not os.path.exists('/content/ComfyUI'):
    !git clone -q https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt
if not os.path.exists('/content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus'):
    !git clone -q https://github.com/cubiq/ComfyUI_IPAdapter_plus /content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus
!pip install -q insightface onnxruntime-gpu opencv-python-headless
print('ComfyUI ready')

In [ ]:
# ── Download RealVis XL 5.0 (photoreal SDXL) + IPAdapter + CLIP-H + InsightFace ──
!mkdir -p /content/ComfyUI/models/checkpoints /content/ComfyUI/models/ipadapter /content/ComfyUI/models/loras /content/ComfyUI/models/clip_vision /content/ComfyUI/models/insightface/models/buffalo_l
CKPT = '/content/ComfyUI/models/checkpoints/realvis_xl_v5.safetensors'
if not os.path.exists(CKPT) or os.path.getsize(CKPT) < 1_000_000_000:
    !rm -f {CKPT}
    !wget --show-progress -O {CKPT} https://huggingface.co/SG161222/RealVisXL_V5.0/resolve/main/RealVisXL_V5.0_fp16.safetensors
if not os.path.exists('/content/ComfyUI/models/ipadapter/ip-adapter-faceid-plusv2_sdxl.bin'):
    !wget -q --show-progress -O /content/ComfyUI/models/ipadapter/ip-adapter-faceid-plusv2_sdxl.bin https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl.bin
    !wget -q --show-progress -O /content/ComfyUI/models/loras/ip-adapter-faceid-plusv2_sdxl_lora.safetensors https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl_lora.safetensors
if not os.path.exists('/content/ComfyUI/models/clip_vision/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors'):
    !wget -q --show-progress -O /content/ComfyUI/models/clip_vision/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors
if not os.path.exists('/content/ComfyUI/models/insightface/models/buffalo_l/det_10g.onnx'):
    !wget -q -O /tmp/buffalo_l.zip https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip
    !unzip -q -o /tmp/buffalo_l.zip -d /content/ComfyUI/models/insightface/models/buffalo_l
!ls -lh /content/ComfyUI/models/checkpoints

In [ ]:
import requests
FRANKIE_REF_URL = 'https://raw.githubusercontent.com/veyrapay/frankie-assets/main/frankie-ref.png'
os.makedirs('/content/ComfyUI/input', exist_ok=True)
with open('/content/ComfyUI/input/frankie-ref.png', 'wb') as f:
    f.write(requests.get(FRANKIE_REF_URL).content)
print('Frankie ref staged')

In [ ]:
# ── Restart ComfyUI server (clean state for new checkpoint) ──
!pkill -f 'main.py' 2>/dev/null; sleep 3
import subprocess, time, requests as r
subprocess.Popen(['python', '/content/ComfyUI/main.py', '--listen', '127.0.0.1', '--port', '8188', '--disable-auto-launch'], cwd='/content/ComfyUI')
for i in range(60):
    try:
        r.get('http://127.0.0.1:8188/system_stats', timeout=2)
        print(f'ComfyUI up after {i*2}s')
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('ComfyUI did not start')
# Verify IPAdapter nodes registered
ni = r.get('http://127.0.0.1:8188/object_info').json()
print('IPAdapterUnifiedLoaderFaceID:', 'IPAdapterUnifiedLoaderFaceID' in ni)
print('IPAdapterFaceID:', 'IPAdapterFaceID' in ni)

In [ ]:
import json, uuid, time, requests as r
POS_PREFIX = 'RAW photo, photograph, photorealistic, sharp focus, 8k uhd, dslr, natural skin texture, film grain, freckles, dark wavy hair, gold hoop earrings, sleeve tattoos, fit body, '
NEGATIVE = 'cartoon, anime, drawing, painting, illustration, 3d render, cgi, deformed, bad anatomy, extra fingers, mutated hands, watermark, text, signature, blurry, low quality, ugly, plastic skin, oversaturated, airbrushed'
SCENES = [
    'topless mirror selfie in bathroom, warm lighting, intimate amateur photo',
    'topless in bed in morning light, smiling, candid amateur photo',
    'wearing black lace lingerie, sitting on bed, sultry, bedroom lighting',
    'in a white t-shirt and panties, kitchen morning, holding coffee mug, candid',
    'topless on a beach at sunset, ocean behind, wet hair, golden hour amateur photo',
    'in a sheer black robe loosely open, balcony at night, city lights',
    'phone selfie in a black string bikini, pool behind, sun-kissed skin, mirror reflection',
    'in a tight white tank top with no bra, gym mirror selfie, sweaty post-workout',
]

def build_workflow(prompt, seed):
    return {
        '1': {'class_type': 'CheckpointLoaderSimple', 'inputs': {'ckpt_name': 'realvis_xl_v5.safetensors'}},
        '2': {'class_type': 'LoadImage', 'inputs': {'image': 'frankie-ref.png'}},
        '3': {'class_type': 'IPAdapterUnifiedLoaderFaceID', 'inputs': {'model': ['1', 0], 'preset': 'FACEID PLUS V2', 'lora_strength': 0.6, 'provider': 'CUDA'}},
        '4': {'class_type': 'IPAdapterFaceID', 'inputs': {'model': ['3', 0], 'ipadapter': ['3', 1], 'image': ['2', 0], 'weight': 0.85, 'weight_faceidv2': 1.0, 'weight_type': 'linear', 'combine_embeds': 'concat', 'start_at': 0.0, 'end_at': 1.0, 'embeds_scaling': 'V only'}},
        '5': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['1', 1]}},
        '6': {'class_type': 'CLIPTextEncode', 'inputs': {'text': NEGATIVE, 'clip': ['1', 1]}},
        '7': {'class_type': 'EmptyLatentImage', 'inputs': {'width': 1024, 'height': 1024, 'batch_size': 1}},
        '8': {'class_type': 'KSampler', 'inputs': {'model': ['4', 0], 'positive': ['5', 0], 'negative': ['6', 0], 'latent_image': ['7', 0], 'seed': seed, 'steps': 30, 'cfg': 6.0, 'sampler_name': 'dpmpp_2m', 'scheduler': 'karras', 'denoise': 1.0}},
        '9': {'class_type': 'VAEDecode', 'inputs': {'samples': ['8', 0], 'vae': ['1', 2]}},
        '10': {'class_type': 'SaveImage', 'inputs': {'images': ['9', 0], 'filename_prefix': 'frankie_real'}},
    }

client_id = str(uuid.uuid4())
results = []
for i, scene in enumerate(SCENES, 1):
    prompt = POS_PREFIX + scene
    wf = build_workflow(prompt, 42 + i * 7)
    print(f'\n[{i}/{len(SCENES)}] {scene}')
    sub = r.post('http://127.0.0.1:8188/prompt', json={'prompt': wf, 'client_id': client_id}).json()
    if 'error' in sub:
        print('  submit error:', json.dumps(sub.get('error'))[:300])
        print('  node_errors:', json.dumps(sub.get('node_errors', {}))[:600])
        continue
    pid = sub['prompt_id']
    img_path = None
    for poll in range(90):
        time.sleep(3)
        h = r.get(f'http://127.0.0.1:8188/history/{pid}').json()
        if pid in h:
            entry = h[pid]
            # Surface any execution error
            status = entry.get('status', {})
            if status.get('status_str') == 'error':
                print('  EXEC ERROR:', json.dumps(status.get('messages', []))[:800])
                break
            outs = entry.get('outputs', {}).get('10', {}).get('images', [])
            if outs:
                f = outs[0]
                img_url = f'http://127.0.0.1:8188/view?filename={f["filename"]}&subfolder={f["subfolder"]}&type={f["type"]}'
                img_path = f'/content/real_{i:02d}.png'
                with open(img_path, 'wb') as fh:
                    fh.write(r.get(img_url).content)
                print(f'  saved {img_path}')
                break
    if img_path:
        results.append((scene, img_path))
    elif not entry.get('status', {}).get('status_str') == 'error':
        print('  timed out (270s)')
print(f'\nDone. {len(results)}/{len(SCENES)} succeeded.')

In [ ]:
from IPython.display import display, Markdown
from PIL import Image
for scene, path in results:
    display(Markdown(f'**{scene}**'))
    display(Image.open(path).resize((512, 512)))

In [ ]:
!cd /content && zip -q frankie_real_batch.zip real_*.png
from google.colab import files
files.download('/content/frankie_real_batch.zip')